In [1]:
!pip install -qU --no-cache-dir datasets huggingface-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.3/509.3 kB 193.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 188.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 180.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2025.3.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.8.4.1 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cudnn-cu12==9.1.0.70; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn-cu12 9.3.0.75 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cufft-cu12

In [2]:
import os
os.environ["HF_HUB_ENABLE_REMOTE_CODE"] = "1"

import datasets, huggingface_hub
print("datasets version:", datasets.__version__)
print("huggingface_hub version:", huggingface_hub.__version__)

datasets version: 3.6.0
huggingface_hub version: 0.32.0


In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
import nltk
import re
from collections import defaultdict, Counter
import pickle
import os
import string
import random
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import spacy
from nltk.corpus import wordnet

# Download resources
nltk.download(['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger'])
spacy.cli.download("en_core_web_sm")

# Configuration
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
CONFIG = {
    "max_sequence_length": 128,
    "embedding_dim": 384,
    "hidden_dim": 512,
    "num_layers": 3,
    "dropout": 0.4,
    "batch_size": 64,
    "epochs": 20,
    "lr": 2e-4,
    "weight_decay": 0.05,
    "gradient_accumulation": 2,
    "focal_alpha": 0.75,
    "focal_gamma": 2,
    "augment_prob": 0.3,
    "subword_ngrams": (3,4),
    "vocab_size": 35000
}

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

# Modified preprocessing function with list handling
def preprocess_text(text):
    """Handle both string and list inputs safely"""
    # Convert list to string if needed
    if isinstance(text, list):
        text = ' '.join(map(str, text))  # Handle any non-string elements
    
    # Ensure text is string type
    text = str(text)
    
    # Rest of preprocessing
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    doc = nlp(text)
    return [token.text for token in doc if not token.is_space]

# Enhanced data loading with validation
def load_data():
    print("Loading datasets...")
    try:
        paws = load_dataset(
        "paws",
        "labeled_final",
        trust_remote_code=True )                        # lets HF run the repo’s custom preprocessing)
        print("PAWS dataset loaded successfully")
    except Exception as e:
        print(f"Error loading PAWS dataset: {str(e)}")
        paws = {"train": [], "validation": [], "test": []}
    
    try:
        quora = load_dataset("quora")['train'].train_test_split(test_size=0.1, seed=SEED)
        print("Quora dataset loaded successfully")
    except Exception as e:
        print(f"Error loading Quora dataset: {str(e)}")
        quora = {"train": [], "test": []}
    
    # Process PAWS data
    try:
        train_df = pd.DataFrame(paws["train"])
        val_df = pd.DataFrame(paws["validation"])
        test_df = pd.DataFrame(paws["test"])
        print(f"PAWS train size: {len(train_df)}, val size: {len(val_df)}, test size: {len(test_df)}")
    except Exception as e:
        print(f"Error processing PAWS data: {str(e)}")
        train_df = pd.DataFrame(columns=["sentence1", "sentence2", "label"])
        val_df = pd.DataFrame(columns=["sentence1", "sentence2", "label"])
        test_df = pd.DataFrame(columns=["sentence1", "sentence2", "label"])
    
    # Process Quora data with validation
    try:
        quora_df = pd.DataFrame(quora["train"])
        
        # Safely extract question pairs
        def get_question_pair(questions, index):
            if questions is None:
                return ""
            if isinstance(questions, list) and len(questions) > index:
                return questions[index] if questions[index] is not None else ""
            return ""
        
        # Extract and clean question pairs
        quora_df['sentence1'] = quora_df['questions'].apply(lambda x: get_question_pair(x, 0))
        quora_df['sentence2'] = quora_df['questions'].apply(lambda x: get_question_pair(x, 1))
        quora_df['label'] = quora_df['is_duplicate'].astype(int)
        
        # Clean data
        quora_df = quora_df[['sentence1', 'sentence2', 'label']].dropna()
        quora_df = quora_df[(quora_df['sentence1'].str.len() > 0) & (quora_df['sentence2'].str.len() > 0)]
        print(f"Quora dataset size after cleaning: {len(quora_df)}")
    except Exception as e:
        print(f"Error processing Quora data: {str(e)}")
        quora_df = pd.DataFrame(columns=["sentence1", "sentence2", "label"])
    
    # Combine datasets
    full_train_df = pd.concat([train_df, quora_df], ignore_index=True)
    print(f"Combined training data size: {len(full_train_df)}")
    
    # Balance dataset if needed
    if len(full_train_df) > 0:
        class_counts = full_train_df['label'].value_counts()
        print(f"Class distribution: {class_counts.to_dict()}")
        
        # Optional: Downsample majority class if severely imbalanced
        if len(class_counts) > 1 and class_counts.max() / class_counts.min() > 10:
            min_class = class_counts.idxmin()
            maj_class = class_counts.idxmax()
            maj_samples = full_train_df[full_train_df['label'] == maj_class].sample(
                n=class_counts[min_class] * 3, random_state=SEED
            )
            min_samples = full_train_df[full_train_df['label'] == min_class]
            full_train_df = pd.concat([maj_samples, min_samples], ignore_index=True)
            print(f"Downsampled to: {len(full_train_df)} samples")
    
    return full_train_df.sample(frac=1, random_state=SEED), val_df, test_df

# Feature engineering
def extract_features(row):
    s1 = row['sentence1_tokens']
    s2 = row['sentence2_tokens']
    
    # Ensure tokens are lists
    if not isinstance(s1, list):
        s1 = []
    if not isinstance(s2, list):
        s2 = []
    
    # N-gram features
    try:
        bigram1 = set(nltk.ngrams(s1, 2)) if len(s1) > 1 else set()
        bigram2 = set(nltk.ngrams(s2, 2)) if len(s2) > 1 else set()
        bigram_overlap = len(bigram1 & bigram2) / (len(bigram1 | bigram2) + 1e-8)
    except Exception:
        bigram_overlap = 0
    
    # POS features
    try:
        pos1 = [tag for _, tag in nltk.pos_tag(s1)]
        pos2 = [tag for _, tag in nltk.pos_tag(s2)]
        pos_jaccard = len(set(pos1) & set(pos2)) / (len(set(pos1) | set(pos2)) + 1e-8)
    except Exception:
        pos_jaccard = 0
    
    # Length features
    len_ratio = abs(len(s1)-len(s2)) / (max(len(s1), len(s2)) + 1e-8)
    
    # Lexical features
    set1, set2 = set(s1), set(s2)
    intersection = set1 & set2
    jaccard = len(intersection) / (len(set1 | set2) + 1e-8)
    containment = len(intersection) / (min(len(set1), len(set2)) + 1e-8) if min(len(set1), len(set2)) > 0 else 0
    
    return [
        jaccard, containment, bigram_overlap, 
        pos_jaccard, len_ratio
    ]

# Subword vocabulary
def build_subword_vocab(df, min_freq=2):
    print("Building subword vocabulary...")
    char_ngrams = defaultdict(int)
    
    # Use both sentence columns
    for field in ['sentence1_tokens', 'sentence2_tokens']:
        # Check if field exists
        if field not in df.columns:
            print(f"Warning: '{field}' not found in dataframe")
            continue
            
        for tokens in df[field]:
            if not isinstance(tokens, list):
                continue
                
            for token in tokens:
                if not isinstance(token, str):
                    continue
                    
                char_ngrams[token] += 1
                for n in CONFIG["subword_ngrams"]:
                    for i in range(len(token)-n+1):
                        char_ngrams[token[i:i+n]] += 1
    
    filtered = {k:v for k,v in char_ngrams.items() if v >= min_freq}
    sorted_items = sorted(filtered.items(), key=lambda x: -x[1])
    vocab = {'<PAD>':0, '<UNK>':1, '<BOS>':2, '<EOS>':3}
    for i, (k,_) in enumerate(sorted_items[:CONFIG["vocab_size"]-4]):
        vocab[k] = i+4
    
    print(f"Vocabulary size: {len(vocab)}")
    return vocab

# Data augmentation
def synonym_replacement(tokens, n=2):
    if not tokens:
        return tokens
        
    if not isinstance(tokens, list):
        return tokens
        
    new_tokens = tokens.copy()
    n = min(n, len(new_tokens))
    
    for _ in range(n):
        if not new_tokens:
            break
            
        idx = random.randint(0, len(new_tokens)-1)
        word = new_tokens[idx]
        
        if not isinstance(word, str):
            continue
            
        syns = get_synonyms(word)
        if syns:
            new_tokens[idx] = random.choice(syns)
            
    return new_tokens

def get_synonyms(word):
    try:
        syns = set()
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                syns.add(lemma.name().replace('_', ' '))
        return list(syns - {word})
    except Exception:
        return []

# Custom collate function
def collate_fn(batch):
    batch_dict = {}
    for key in batch[0].keys():
        if key in ['seq1', 'seq2', 'features', 'label']:
            # These are tensors and can be stacked
            batch_dict[key] = torch.stack([item[key] for item in batch])
        else:
            # These are scalars and need to be converted to tensors
            batch_dict[key] = torch.tensor([item[key] for item in batch])
    return batch_dict

# Dataset class
class ParaphraseDataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx].copy()
        
        # Data augmentation for positive examples
        if row['label'] == 1 and random.random() < CONFIG["augment_prob"]:
            try:
                new_tokens = synonym_replacement(row['sentence1_tokens'])
                row['sentence2_tokens'] = new_tokens
            except Exception:
                pass  # Skip augmentation if it fails
        
        # Ensure tokens exist
        if 'sentence1_tokens' not in row or not isinstance(row['sentence1_tokens'], list):
            row['sentence1_tokens'] = []
        if 'sentence2_tokens' not in row or not isinstance(row['sentence2_tokens'], list):
            row['sentence2_tokens'] = []
        
        seq1, len1 = self.tokenize(row['sentence1_tokens'])
        seq2, len2 = self.tokenize(row['sentence2_tokens'])
        
        return {
            'seq1': seq1,
            'seq2': seq2,
            'len1': len1,
            'len2': len2,
            'features': torch.FloatTensor(extract_features(row)),
            'label': torch.LongTensor([row['label']])
        }
    
    def tokenize(self, tokens):
        # Handle empty or invalid tokens
        if not tokens:
            indices = []
        else:
            indices = [self.vocab.get(t, 1) for t in tokens[:CONFIG["max_sequence_length"]-2]]
            
        # Add BOS and EOS tokens
        indices = [self.vocab['<BOS>']] + indices + [self.vocab['<EOS>']]
        actual_len = len(indices)
        
        # Pad sequence
        indices += [0] * (CONFIG["max_sequence_length"] - len(indices))
        return torch.LongTensor(indices), actual_len

# Model architecture
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.LayerNorm(dim),
            nn.Dropout(0.3)
        )
        
    def forward(self, x):
        return x + self.block(x)

class EnhancedSiameseLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, CONFIG["embedding_dim"], padding_idx=0)
        self.emb_drop = nn.Dropout(CONFIG["dropout"])
        
        self.lstm = nn.LSTM(
            CONFIG["embedding_dim"], CONFIG["hidden_dim"],
            num_layers=CONFIG["num_layers"],
            bidirectional=True,
            dropout=CONFIG["dropout"] if CONFIG["num_layers"] > 1 else 0,
            batch_first=True
        )
        self.lstm_ln = nn.LayerNorm(CONFIG["hidden_dim"]*2)
        
        self.attention = nn.MultiheadAttention(
            CONFIG["hidden_dim"]*2, num_heads=4,
            dropout=CONFIG["dropout"], batch_first=True
        )
        
        self.feature_net = nn.Sequential(
            nn.Linear(5, 128),
            nn.ReLU(),
            nn.LayerNorm(128),
            nn.Dropout(CONFIG["dropout"])
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(CONFIG["hidden_dim"]*8 + 128, 1024),
            nn.ReLU(),
            nn.LayerNorm(1024),
            nn.Dropout(0.3),
            ResidualBlock(1024),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.LayerNorm(512),
            nn.Dropout(0.3),
            nn.Linear(512, 2)
        )

    def forward(self, seq1, seq2, len1, len2, features):
        enc1 = self.encode(seq1, len1)
        enc2 = self.encode(seq2, len2)
        
        feat_emb = self.feature_net(features)
        
        combined = torch.cat([
            enc1, enc2,
            torch.abs(enc1 - enc2),
            enc1 * enc2,
            feat_emb
        ], dim=1)
        
        return self.classifier(combined)
    
    def encode(self, x, lengths):
        embedded = self.emb_drop(self.embedding(x))
        
        # Handle zero-length sequences
        if torch.all(lengths == 0):
            batch_size = x.size(0)
            return torch.zeros(batch_size, CONFIG["hidden_dim"]*2, device=x.device)
            
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(output, batch_first=True, total_length=CONFIG["max_sequence_length"])
        output = self.lstm_ln(output)
        
        # Apply attention
        mask = torch.arange(CONFIG["max_sequence_length"], device=x.device)[None, :] < lengths[:, None]
        attn_output, _ = self.attention(output, output, output, key_padding_mask=~mask)
        
        # Apply mean pooling
        return torch.sum(attn_output * mask.unsqueeze(-1), dim=1) / (lengths.unsqueeze(-1) + 1e-8)

# Focal loss
class FocalLoss(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(reduction="none")(inputs, targets.squeeze())
        pt = torch.exp(-ce_loss)
        loss = (CONFIG["focal_alpha"] * (1-pt)**CONFIG["focal_gamma"] * ce_loss).mean()
        return loss

# Training loop
def train():
    # Load data
    train_df, val_df, test_df = load_data()
    
    # Ensure datasets are not empty
    if len(train_df) == 0:
        print("Error: Training dataset is empty")
        return
    if len(val_df) == 0:
        print("Warning: Validation dataset is empty. Creating from training data...")
        train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=SEED)
    if len(test_df) == 0:
        print("Warning: Test dataset is empty. Creating from training data...")
        train_df, test_df = train_test_split(train_df, test_size=0.1, random_state=SEED)
    
    # Preprocessing
    print("Preprocessing data...")
    for df in [train_df, val_df, test_df]:
        df['sentence1_tokens'] = df['sentence1'].apply(preprocess_text)
        df['sentence2_tokens'] = df['sentence2'].apply(preprocess_text)
    
    # Build vocab
    vocab = build_subword_vocab(train_df)
    
    # Create datasets
    train_dataset = ParaphraseDataset(train_df, vocab)
    val_dataset = ParaphraseDataset(val_df, vocab)
    test_dataset = ParaphraseDataset(test_df, vocab)
    
    # Data loaders
    train_loader = DataLoader(
        train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,
        collate_fn=collate_fn
    )
    val_loader = DataLoader(
        val_dataset, batch_size=CONFIG["batch_size"],
        collate_fn=collate_fn
    )
    test_loader = DataLoader(
        test_dataset, batch_size=CONFIG["batch_size"],
        collate_fn=collate_fn
    )
    
    # Model setup
    model = EnhancedSiameseLSTM(len(vocab)).to(device)
    
    # Print model summary
    print(f"Model Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=CONFIG["lr"]*10,
        steps_per_epoch=len(train_loader) // CONFIG["gradient_accumulation"] + 1,
        epochs=CONFIG["epochs"]
    )
    criterion = FocalLoss()
    
    # Model training
    print("\nStarting training...")
    best_f1 = 0
    for epoch in range(CONFIG["epochs"]):
        model.train()
        train_loss = 0
        preds, labels = [], []
        
        for batch_idx, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}")):
            # Move tensors to device
            inputs = {k: v.to(device) for k, v in batch.items() if k != 'label'}
            targets = batch['label'].to(device)
            
            # Forward pass
            outputs = model(**inputs)
            loss = criterion(outputs, targets)
            
            # Scale loss for gradient accumulation
            loss = loss / CONFIG["gradient_accumulation"]
            loss.backward()
            
            # Update weights with accumulation
            if (batch_idx + 1) % CONFIG["gradient_accumulation"] == 0 or (batch_idx + 1 == len(train_loader)):
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
            
            # Track metrics
            train_loss += loss.item() * CONFIG["gradient_accumulation"]
            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            labels.extend(targets.cpu().numpy())
        
        # Validation phase
        model.eval()
        val_loss = 0
        val_preds, val_labels = [], []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                # Move tensors to device
                inputs = {k: v.to(device) for k, v in batch.items() if k != 'label'}
                targets = batch['label'].to(device)
                
                # Forward pass
                outputs = model(**inputs)
                loss = criterion(outputs, targets)
                
                # Track metrics
                val_loss += loss.item()
                val_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
                val_labels.extend(targets.cpu().numpy())
        
        # Calculate metrics
        train_acc = accuracy_score(labels, preds)
        train_f1 = f1_score(labels, preds, average='weighted')
        val_acc = accuracy_score(val_labels, val_preds)
        val_f1 = f1_score(val_labels, val_preds, average='weighted')
        
        # Print metrics
        print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}")
        print(f"Train Loss: {train_loss/len(train_loader):.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
        print(f"Val Loss: {val_loss/len(val_loader):.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
        
        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_f1': val_f1,
                'vocab': vocab
            }, "best_model.pth")
            print("Saved new best model")
    
    # Final evaluation on test set
    print("\nLoading best model for final evaluation...")
    checkpoint = torch.load("best_model.pth")
    model.load_state_dict(checkpoint['model_state_dict'])
    
    test_preds, test_labels = [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            inputs = {k: v.to(device) for k, v in batch.items() if k != 'label'}
            outputs = model(**inputs)
            test_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            test_labels.extend(batch['label'].cpu().numpy())
    
    # Final metrics
    test_acc = accuracy_score(test_labels, test_preds)
    test_f1 = f1_score(test_labels, test_preds, average='weighted')
    
    print("\nFinal Test Results:")
    print(f"Accuracy: {test_acc:.4f}")
    print(f"F1 Score: {test_f1:.4f}")
    print(classification_report(test_labels, test_preds))
    
    # Plot confusion matrix
    cm = confusion_matrix(test_labels, test_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.savefig('confusion_matrix.png')
    plt.close()
    
    print("Confusion Matrix saved as 'confusion_matrix.png'")
    
    # Save test metrics
    test_results = {
        'accuracy': test_acc,
        'f1_score': test_f1,
        'classification_report': classification_report(test_labels, test_preds, output_dict=True),
        'confusion_matrix': cm.tolist()
    }
    
    with open('test_results.pkl', 'wb') as f:
        pickle.dump(test_results, f)
    
    print("Test results saved to 'test_results.pkl'")

if __name__ == "__main__":
    # Import needed for train/test split
    from sklearn.model_selection import train_test_split
    train()

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 36.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Using device: cuda
Loading datasets...


README.md:   0%|          | 0.00/9.79k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/8.43M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.24M [00:00<?, ?B/s]

...

README.md:   0%|          | 0.00/5.69k [00:00<?, ?B/s]

quora.py:   0%|          | 0.00/2.38k [00:00<?, ?B/s]

The repository for quora contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/quora.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Generating train split:   0%|          | 0/404290 [00:00<?, ? examples/s]

Quora dataset loaded successfully
PAWS train size: 49401, val size: 8000, test size: 8000
